In [2]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [3]:
from src.ingestion.musicbrainz_client import MusicBrainzClient
import json

In [4]:
mb_client = MusicBrainzClient()

In [5]:
for name in ["Nirvana", "Smashing Pumpkins", "Pearl Jam", "Soundgarden", "Blur", "Deftones"]:
    search_result = mb_client.search_artist(name)
    top_match = search_result['artists'][0]
    print(f"{name} -> {top_match['name']} (score: {top_match['score']}, id: {top_match['id']})")

https://musicbrainz.org/ws/2/artist
Nirvana -> Nirvana (score: 100, id: 5b11f4ce-a62d-471e-81fc-a69a8278c7da)
https://musicbrainz.org/ws/2/artist
Smashing Pumpkins -> The Smashing Pumpkins (score: 100, id: ba0d6274-db14-4ef5-b28d-657ebde1a396)
https://musicbrainz.org/ws/2/artist
Pearl Jam -> Pearl Jam (score: 100, id: 83b9cbe7-9857-49e2-ab8e-b57b01038103)
https://musicbrainz.org/ws/2/artist
Soundgarden -> Soundgarden (score: 100, id: 153c9281-268f-4cf3-8938-f5a4593e5df4)
https://musicbrainz.org/ws/2/artist
Blur -> Blur (score: 100, id: ba853904-ae25-4ebb-89d6-c44cfbd71bd2)
https://musicbrainz.org/ws/2/artist
Deftones -> Deftones (score: 100, id: 7527f6c2-d762-4b88-b5e2-9244f1e34c46)


In [6]:
seed_ids = {
    "5b11f4ce-a62d-471e-81fc-a69a8278c7da": "Nirvana",
    "ba0d6274-db14-4ef5-b28d-657ebde1a396": "The Smashing Pumpkins",
    "83b9cbe7-9857-49e2-ab8e-b57b01038103": "Pearl Jam",
    "153c9281-268f-4cf3-8938-f5a4593e5df4": "Soundgarden",
    "ba853904-ae25-4ebb-89d6-c44cfbd71bd2": "Blur",
    "7527f6c2-d762-4b88-b5e2-9244f1e34c46": "Deftones",        
}

raw_dir = "../data/raw/artists"

for filename in os.listdir(raw_dir):
    filepath = os.path.join(raw_dir, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        artist_data = json.load(f)

    for rel in artist_data.get("relations", []):
        target = rel.get("artist")
        if target and target["id"] in seed_ids:
            print(f"{artist_data['name']} --[{rel['type']}]--> {target['name']}")

In [7]:
from collections import defaultdict

person_to_bands = defaultdict(set)

for filename in os.listdir(raw_dir):
    filepath = os.path.join(raw_dir, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        artist_data = json.load(f)

    band_name = artist_data['name']

    for rel in artist_data.get("relations", []):
        target = rel.get("artist")
        if target and target.get("type") == "Person":
            person_to_bands[target["name"]].add(band_name)

for person, bands in person_to_bands.items():
    if len(bands) > 1:
        print(f"{person} -> {bands}")

Matt Cameron -> {'Soundgarden', 'Pearl Jam'}
Jason Everman -> {'Soundgarden', 'Nirvana'}


In [8]:
import pandas as pd

raw_dir = "../data/raw/artists"
rows = []

for filename in os.listdir(raw_dir):
    filepath = os.path.join(raw_dir, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        artist = json.load(f)

    tags = artist.get("tags", [])
    top_tags = sorted(tags, key=lambda t: t["count"], reverse=True)[:3]
    top_tag_names = [t["name"] for t in top_tags]

    rows.append({
        "mbid": artist.get("id"),
        "name": artist.get("name"),
        "type": artist.get("type"),
        "country": artist.get("country"),
        "begin_date": artist.get("life-span", {}).get("begin"),
        "num_relations": len(artist.get("relations", [])),
        "num_release_groups": len(artist.get("release-groups", [])),
        "top_tags": ", ".join(top_tag_names)
    })

df = pd.DataFrame(rows)
df

,mbid,name,type,country,begin_date,num_relations,num_release_groups,top_tags
0,153c9281-268f-4cf3-8938-f5a4593e5df4,Soundgarden,Group,US,1984,13,25,"grunge, alternative rock, alternative metal"
1,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,Group,US,1987,45,25,"grunge, alternative rock, rock"
2,7527f6c2-d762-4b88-b5e2-9244f1e34c46,Deftones,Group,US,1988,10,25,"alternative metal, nu metal, alternative rock"
3,83b9cbe7-9857-49e2-ab8e-b57b01038103,Pearl Jam,Group,US,1990,41,25,"grunge, alternative rock, rock"
4,ba0d6274-db14-4ef5-b28d-657ebde1a396,The Smashing Pumpkins,Group,US,1988,21,25,"alternative rock, dream pop, neo-psychedelia"
5,ba853904-ae25-4ebb-89d6-c44cfbd71bd2,Blur,Group,GB,1990,13,25,"alternative rock, britpop, indie rock"


In [9]:
print(df['name'].tolist())

['Soundgarden', 'Nirvana', 'Deftones', 'Pearl Jam', 'The Smashing Pumpkins', 'Blur']


In [12]:
from src.preprocessing.release_preprocessor import release_to_track_rows, pick_best_release

all_rows = []

seed_artist_map = {
    "Nirvana": "5b11f4ce-a62d-471e-81fc-a69a8278c7da",
    "The Smashing Pumpkins": "ba0d6274-db14-4ef5-b28d-657ebde1a396",
    "Pearl Jam": "83b9cbe7-9857-49e2-ab8e-b57b01038103",
    "Soundgarden": "153c9281-268f-4cf3-8938-f5a4593e5df4",
    "Blur": "ba853904-ae25-4ebb-89d6-c44cfbd71bd2",
    "Deftones": "7527f6c2-d762-4b88-b5e2-9244f1e34c46",
}

for artist_name, artist_id in seed_artist_map.items():
    try:
        with open(f"../data/raw/artists/{artist_id}.json", "r", encoding="utf-8") as f:
            artist_data = json.load(f)

        clean_albums = [
            rg for rg in artist_data["release-groups"]
            if rg.get("primary-type") == "Album" and not rg.get("secondary-types")
        ]

        if not clean_albums:
            print(f"{artist_name}: no clean albums, skipping")
            continue

        chosen_album = clean_albums[0]

        releases = mb_client.get_releases_from_group(chosen_album["id"])
        best_release = pick_best_release(releases)

        if best_release is None:
            print(f"{artist_name}: no suitable release found, skipping")
            continue

        release_detail = mb_client.get_release(best_release["id"], inc="recordings+media+artist-credits+labels")

        rows = release_to_track_rows(
            release_detail=release_detail,
            artist_id=artist_id,
            artist_name=artist_name,
            release_group_id=chosen_album["id"],
            release_group_title=chosen_album["title"]
        )

        all_rows.extend(rows)
        print(f"{artist_name}: {len(rows)} tracks added from '{chosen_album['title']}' ({best_release.get('date')})")

    except Exception as e:
        print(f"{artist_name}: FAILED - {e}")
        continue

album_tracks_df = pd.DataFrame(all_rows)
album_tracks_df

Nirvana: 13 tracks added from 'Bleach' (1989)
The Smashing Pumpkins: FAILED - HTTPSConnectionPool(host='musicbrainz.org', port=443): Read timed out. (read timeout=30)
Pearl Jam: 11 tracks added from 'Ten' (1991)
Soundgarden: 13 tracks added from 'Ultramega OK' (1988-10-31)
Blur: 12 tracks added from 'Leisure' (1991)
Deftones: 11 tracks added from 'Adrenaline' (1995-10-02)


,artist_id,artist_name,release_group_id,release_group_title,release_id,release_title,release_date,country,medium_format,track_position,track_title,recording_id,length_ms
0,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,1,Blew,2837e0d2-5ce9-44da-803e-797c835d672c,174133.0
1,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,2,Floyd the Barber,4cce1ee1-38e6-4ea0-8584-b6ebeb854d7d,137973.0
2,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,3,About a Girl,b2e623ea-378b-4479-bb75-fa202aacbfc9,168293.0
3,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,4,School,0b3292f6-8f0d-4160-92d3-49348029a619,162173.0
4,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,5,Love Buzz,1adfecb7-875f-4203-b3b1-8e2e643f94a2,215093.0
5,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,6,Paper Cuts,67930d52-5e75-4c00-b36f-ee3c43bed160,245840.0
6,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,7,Negative Creep,06745db3-a2bc-4131-8a9d-0cbdc25b494a,175693.0
7,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,8,Scoff,dbecedb1-2a90-43fc-add4-ce4016676545,250133.0
8,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,9,Swap Meet,c9972a41-2ae8-43f6-9822-ba2fec7e50d1,182973.0
9,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,10,Mr. Moustache,6cd0083c-e4ce-4745-aaf0-704ab1de988a,203893.0


In [16]:
import time
time.sleep(15)

sp_id = "ba0d6274-db14-4ef5-b28d-657ebde1a396"

with open(f"../data/raw/artists/{sp_id}.json", "r", encoding="utf-8") as f:
    sp_data = json.load(f)

clean_albums = [
    rg for rg in sp_data["release-groups"]
    if rg.get("primary-type") == "Album" and not rg.get("secondary-types")
]
chosen_album = clean_albums[0]

releases = mb_client.get_releases_from_group(chosen_album["id"])
best_release = pick_best_release(releases)

release_detail = mb_client.get_release(best_release["id"], inc="recordings+media+artist-credits+labels")

rows = release_to_track_rows(
    release_detail=release_detail,
    artist_id=sp_id,
    artist_name="The Smashing Pumpkins",
    release_group_id=chosen_album["id"],
    release_group_title=chosen_album["title"]
)

all_rows.extend(rows)
print(f"The Smashing Pumpkins: {len(rows)} tracks added from '{chosen_album['title']}' ({best_release.get('date')})")

The Smashing Pumpkins: 10 tracks added from 'Gish' (1991)


In [17]:
print(len(all_rows))

70


In [18]:
album_tracks_df = pd.DataFrame(all_rows)
album_tracks_df.to_csv("../data/processed/album_tracks.csv", index=False)
album_tracks_df

,artist_id,artist_name,release_group_id,release_group_title,release_id,release_title,release_date,country,medium_format,track_position,track_title,recording_id,length_ms
0,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,1,Blew,2837e0d2-5ce9-44da-803e-797c835d672c,174133.0
1,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,2,Floyd the Barber,4cce1ee1-38e6-4ea0-8584-b6ebeb854d7d,137973.0
2,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,3,About a Girl,b2e623ea-378b-4479-bb75-fa202aacbfc9,168293.0
3,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,4,School,0b3292f6-8f0d-4160-92d3-49348029a619,162173.0
4,5b11f4ce-a62d-471e-81fc-a69a8278c7da,Nirvana,f1afec0b-26dd-3db5-9aa1-c91229a74a24,Bleach,02b23593-ad18-4c8c-b30a-5c96b4796289,Bleach,1989,CA,CD,5,Love Buzz,1adfecb7-875f-4203-b3b1-8e2e643f94a2,215093.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,ba0d6274-db14-4ef5-b28d-657ebde1a396,The Smashing Pumpkins,4ac82f0b-1532-3734-80d7-5c02265197aa,Gish,4009d8ba-5f3d-3172-ab8b-1df53d83bfa6,Gish,1991,NL,CD,6,Suffer,d283e46e-28e3-4d0c-96f0-4c67089a97a3,311266.0
66,ba0d6274-db14-4ef5-b28d-657ebde1a396,The Smashing Pumpkins,4ac82f0b-1532-3734-80d7-5c02265197aa,Gish,4009d8ba-5f3d-3172-ab8b-1df53d83bfa6,Gish,1991,NL,CD,7,Snail,d611c71c-0551-4083-abfe-89426c80dab4,311000.0
67,ba0d6274-db14-4ef5-b28d-657ebde1a396,The Smashing Pumpkins,4ac82f0b-1532-3734-80d7-5c02265197aa,Gish,4009d8ba-5f3d-3172-ab8b-1df53d83bfa6,Gish,1991,NL,CD,8,Tristessa,90c44add-190e-4cc6-89db-ae9b9aeea745,213600.0
68,ba0d6274-db14-4ef5-b28d-657ebde1a396,The Smashing Pumpkins,4ac82f0b-1532-3734-80d7-5c02265197aa,Gish,4009d8ba-5f3d-3172-ab8b-1df53d83bfa6,Gish,1991,NL,CD,9,Window Paine,b3b0764b-9ccb-43fd-bb61-37abdcf0c051,351706.0
